![logo](../../.././docs/images/Logo_Destination_Earth_Colours.png)

# Solution 1: Basic Data Retrieval from Climate DT

In this notebook we will pull basic full-field data from the **Climate Digital Twin**. The Climate DT provides climate projection data on a HEALPix grid, which we will retrieve, inspect, plot, and convert to xarray.

## Authentication

In [ ]:
%%capture cap
%run ../../desp-authentication.py

In [ ]:
output_1 = cap.stdout.split('}\n')
access_token = output_1[-1][0:-1]

## Imports

In [ ]:
import earthkit.data
import earthkit.plots
import earthkit.regrid
import os

## Live Request Toggle

In [ ]:
LIVE_REQUEST = os.getenv("LIVE_REQUEST", "true").lower() == "true"
LIVE_REQUEST

## Build the Request

We request 2m temperature and 10m U wind from the Climate DT projections dataset.

In [ ]:
request = {
    "activity": "projections",
    "class": "d1",
    "dataset": "climate-dt",
    "date": "20400615",
    "experiment": "SSP3-7.0",
    "expver": "0001",
    "generation": "2",
    "levtype": "sfc",
    "model": "IFS-NEMO",
    "param": "167/165",
    "realization": "1",
    "resolution": "standard",
    "stream": "clte",
    "time": "1200",
    "type": "fc",
}

## Retrieve the Data

In [ ]:
data_file = "../../climate-dt/data/climate-dt-basic-training.grib"

if LIVE_REQUEST:
    data = earthkit.data.from_source(
        "polytope", "destination-earth", request,
        address="polytope.mn5.apps.dte.destination-earth.eu",
        stream=False,
    )
    data.to_target("file", data_file)
else:
    data = earthkit.data.from_source("file", data_file)

## Inspect the Data

Use `.ls()` to inspect the fields returned.

In [ ]:
data.ls()

## Plot the Data

In [ ]:
chart = earthkit.plots.Map(extent=[-180, 180, -90, 90])
chart.grid_cells(data, style="auto")

chart.coastlines()
chart.gridlines()
chart.title("Climate DT - 2m Temperature & 10m U Wind (Jun 2040)")

chart.show()

## Convert to xarray

To work with the data in xarray, we first regrid from HEALPix to a regular lat-lon grid.

In [ ]:
data_latlon = earthkit.regrid.interpolate(data, out_grid={"grid": [1, 1]}, method="linear")
ds = data_latlon.to_xarray()
ds